In [31]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "martin2010keeping")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "martinordas2010keeping_exp 1_episodic-like_memory.csv")
complete_path_2 = os.path.join(original_data_pathway, "martinordas2010keeping_exp 2_episodic-like_memory.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [32]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_csv(complete_path_1)
df1 = df1.assign(experiment='1')
df2 = pd.read_csv(complete_path_2)
df2 = df2.assign(experiment='2')
# df1.columns


In [33]:
r1 = df1[['ape', 'time', 'total', '1_round_ice_cube', '1_round_grape', 'experiment']]
r1 = r1.assign(round='1')
r2 = df1[['ape', 'time', 'total', '2_round_ice_cube', '2_round_grape', 'experiment']]
r2 = r2.assign(round='2')
r3 = df1[['ape', 'time', 'total', '3_round_ice_cube', '3_round_grape', 'experiment']]
r3 = r3.assign(round='3')

r_frames = [r1, r2, r3]
for index, x in enumerate(r_frames):
    x.rename(columns={"1_round_ice_cube": "ice_cube",
        "1_round_grape":"grape",
        "2_round_ice_cube":"ice_cube",
        "2_round_grape":"grape",
        "3_round_ice_cube": "ice_cube",
        "3_round_grape":"grape"}, inplace=True)
    r_frames[index]=x
new_df=r_frames[0]
fullr = pd.concat(r_frames, ignore_index=True, sort=False)

In [34]:
data_frames=[fullr, df2]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x['study_id']="martin2010keeping"
    x['ape'] = x['ape'].str.rstrip()
    data_frames[index]=x
new_df=data_frames[0]
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)


In [35]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['ape'] = fulldf['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left')


In [36]:
# fulldf.columns
fulldf.rename(columns={"ape": "participant"}, inplace=True)

complete_path_age = os.path.join(original_data_pathway, "subject_list.csv")
subject_list = pd.read_csv(complete_path_age)   
fulldf= fulldf.merge(subject_list,left_on='participant', right_on='name', how='left')
fulldf.rename(columns={"age": "age_in_years",
                       'trial ':"trial"}, inplace=True)


# fulldf.columns
fulldf['trial'].replace('trial ', '', inplace=True, regex = True)

In [37]:
fulldf=fulldf[['study_id','experiment', 'participant','age_in_years', 'sex','species', 'trial', 'round',
        'time',  'ice_cube', 'grape',
       '5_min_ice_cube', '5_min_grape', '1_hour_ice_cube', '1_hour_grape']]


In [38]:
for index in range(1,3):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'martin2010keeping_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'martin2010keeping_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)

